# BME 590 - Workshop 0 - Setup & Introduction
**Professor:** Emma Chory, Ph.D.

**Authors:** 
Rick Wierenga, Joe Laforet, Stefan Golas, Ben Perry

---

---

#### ⚠️ Check your kernel first (top-right corner)

It should read **BME 590 (lab automation)**. If it says *“Select Kernel”*, a Python version, or anything else:

1. Click the kernel picker in the top-right.
2. Choose **Select Another Kernel → Jupyter Kernel...**
3. Pick **BME 590 (lab automation)** — scroll past the first group; it sits under its own heading, one level down.

Wrong kernel = `ModuleNotFoundError` / *"requires the ipykernel package"* on the very first cell. The fix is the 10 seconds above — **do not pip-install anything**.

---


In [ ]:
# How long the visualizer pauses between steps, as a multiple of the
# workshop's original timing. The pauses are there so you can watch the
# deck update -- once you have seen it, set this to 0 to run at full speed.
SLEEP = 1.0


### Usage Note
**Reminder** - You should be running this notebook **locally** on **VS Code** not navigating it through **GitHub**.

---

### Welcome to PyLabRobot!
PyLabRobot *(PLR)* is a universal Python hardware and operating system-agnostic software development kit for automated and autonomous laboratories. PyLabRobot enables control of liquid handling robots, plate readers, pumps, scales, heater shakers, and other equiprment by converting Python commands to their corresponding low-level firmware/IO commands. The primary piece of equipment we care about controlling is the liquid handler, which is a robot that can aspirate and dispense precise volumes of liquid in a Cartesian coordinate system, essentially the same as hand pipetting, but automated!

PLR defines a several universal interface classes:

- **LiquidHandler:** provides generic methods for controlling liquid handlers

- **PlateReader:** (and some other classes) control other equipment like a plate reader. 

These **interface classes** are able to translate singular (**atomic**) commands for a robot (aspirate, drop tips, dispense, move, etc.) to any number of **supported robots** via custom **backends** (or drivers). \[See Figure 1 for a diagram of how this works\] This setup enables the definition of a protocol in Python, subsequent translation to any number of robots, all in one Python script or notebook!

<div>
<img src="../figs/fig_1_plr.jpg" width="1000"/>
</div>

As we work through exercises and tutorials for PLR, make note of these specific help resources! Specifically, reading the **publication below** will help you get oriented with the organization of PLR. For more specific questions, reach out on the class **slack** or to the **TA** or **Professor** via email.

**PLR GitHub:** [Link](https://github.com/PyLabRobot/pylabrobot)

**PLR Forum:** [Link](https://discuss.pylabrobot.org/)

**Publication:** [Link](https://www.cell.com/device/fulltext/S2666-9986\(23\)00170-9)

### Getting Started

---

**NOTE:** If you already have PLR installed and working on your local machine, you can **skip to the Setting Up Your First Deck & Visualizer section**

---

#### PLR Installation
To get started with PyLabRobot in a research environment, you should follow the [installation instructions](https://docs.pylabrobot.org/user_guide/_getting-started/installation.html) in the online documentation, 

However, for BME 590, we provide a more robust set of installation instructions to ensure the entire class is on the same page to start. These instructions can be found on the class repository located [here](https://github.com/chory-lab/bme590-fall-2026).

Please run through these instructions first, then return to this notebook. Technically, if you are seeing this notebook, you have at least partially gotten through those instructions so congrats!

Let's go ahead and test your PLR installation! You should have an output that appears like:

```txt
PLR Version:  0.2.2
```

In [ ]:
import pylabrobot

print("PLR Version: ", pylabrobot.__version__)

#### Auto-complete / Pylance Missing Imports Issue

Running `import pylabrobot` works, but VS Code underlines it as missing (a Pylance squiggle)? That means the notebook is using the wrong Python environment. The install wrote `.vscode/settings.json` to point at this project's `.venv` and registered the **BME 590 (lab automation)** kernel, so this should already be wired up.

If you still see the squiggle:

1. Check the kernel. In any notebook, the kernel picker (top-right, or the one VS Code offers on first open) should show **BME 590 (lab automation)**.

2. Check the interpreter. Click the Python version in the bottom-right status bar and choose the one inside this project's `.venv`.

3. If it still fails, run `uv run bme590 check` from the class folder in a terminal and paste its **entire** output into the `#ed-discuss` Slack channel.


#### Setting Up Your First Deck & Visualizer

Let's go ahead and set up your first deck for a liquid handler. To set up a liquid handler, you will need three things:

- **Liquid Handler Interface** - This is the top level class that will organize everything else for us.

- **A Deck Layout** - Different machines have different deck sizes/slots. The deck layout tells the LiquidHandler the geometric constraints of our robot.

- **A Backend** -- The backend does the heavy lifting of converting the instructions we write in Python to **machine code** tailored to each robot. Because we do not have a lab component to this class (yet), we must do our experimentation *virtually*. Fortunately, PLR has a built-in `Visualizer` class, which enables us to host a website locally that will display our active experiment as we work!

In the later workshops we will get into the specifics of setting up a deck, and different types of liquid handlers that are available through PLR. For now, we are just trying to make sure you can:

1. Setup a liquid handler deck.

2. Setup the visualizer.

3. Record a GIF of a basic protocol.

Let's start with setting up a deck. Run the following code to set up your first deck. Again, doon't worry about the following code; next week's workshop will cover it in detail.

In [ ]:
from pylabrobot.resources import (STARLetDeck, TIP_CAR_480_A00, PLT_CAR_L5AC_A00, cor_96_wellplate_360uL_Fb, hamilton_96_tiprack_1000uL_filter)

async def make_deck_with_carriers_and_contents():
    deck = STARLetDeck()

    # create carriers
    tip_carrier = TIP_CAR_480_A00(name="awesome tip carrier 96x5")
    plate_carrier = PLT_CAR_L5AC_A00(name = "awesome plate carrier")

    # assign carriers
    deck.assign_child_resource(plate_carrier, rails = 5)
    deck.assign_child_resource(tip_carrier, rails = 11)

    # define 2 tip racks with unique names and assign to the first two slots of the tip carrier
    for i in range(3):
        tip_carrier[i] = hamilton_96_tiprack_1000uL_filter(name=f"tip_rack_{i}")

    # define 4 plates with unique names and assign to the first two slots of the plate carrier
    for i in range(4):
        plate_carrier[i] = cor_96_wellplate_360uL_Fb(name=f"plate_{i}")
        
    return deck

deck = await make_deck_with_carriers_and_contents()

Great, you should now have a deck variable containing all the information of your first deck. Let's print a summary of this information.

In [ ]:
print(deck.summary())

Great! You should get an output that looks like this:

```txt
Rail  Resource                        Type           Coordinates (mm)
===================================================================================
(-6)  ├── trash_core96                Trash          (-58.200, 106.000, 229.000)
      │
(5)   ├── awesome plate carrier       PlateCarrier   (190.000, 063.000, 100.000)
      │   ├── plate_0                 Plate          (194.000, 071.500, 183.120)
      │   ├── plate_1                 Plate          (194.000, 167.500, 183.120)
      │   ├── plate_2                 Plate          (194.000, 263.500, 183.120)
      │   ├── plate_3                 Plate          (194.000, 359.500, 183.120)
      │   ├── <empty>
      │
(11)  ├── awesome tip carrier 96x5    TipCarrier     (325.000, 063.000, 100.000)
      │   ├── tip_rack_0              TipRack        (331.200, 073.000, 214.950)
      │   ├── tip_rack_1              TipRack        (331.200, 169.000, 214.950)
      │   ├── <empty>
      │   ├── <empty>
      │   ├── <empty>
      │
(31)  ├── waste_block                 Resource       (775.000, 115.000, 100.000)
      │   ├── teaching_tip_rack       TipRack        (780.900, 461.100, 100.000)
      │
(32)  ├── trash                       Trash          (800.000, 190.600, 137.100)
```

But this is kind of hard to actually visualize what is going on. To do that, let's setup our `Visualizer`

In [ ]:
from pylabrobot.resources import Deck
from pylabrobot.liquid_handling.backends.backend import LiquidHandlerBackend
from pylabrobot.liquid_handling.backends import LiquidHandlerChatterboxBackend
from pylabrobot.visualizer.visualizer import Visualizer
from pylabrobot.liquid_handling import LiquidHandler

async def visualize_deck(deck: Deck,
                         backend: LiquidHandlerBackend):
    try:
        lh = LiquidHandler(backend=backend, deck=deck)
        vis = Visualizer(resource = lh)
        await lh.setup()
        await vis.setup()
        return lh
    except Exception as e:
        print(f"Error! Got excpetion: {e}")

lh = await visualize_deck(deck, LiquidHandlerChatterboxBackend())

Running the above code should output something similar to:

```txt
Websocket server started at http://127.0.0.1:XXXX
File server started at http://127.0.0.1:XXXX . Open this URL in your browser.
```

This is effectively hosting a website at 127.0.0.1, which is the **localhost**, a special IP address that means the server is being hosted on your own machine. This means when you open the URL provided in the output, you are **connecting** to your own machine's server. You should see a chrome (or whatever default browser your machine has) tab open with an output that is similar to this:

<div>
<img src="../figs/carrier_layout.png" width="750"/>
</div>

We aren't giving you the exact photo of the setup, because you will need to submit that as part of your assignment!

If a browser doesn't automatically open, simply **copy/paste** the output URL to your browser of choice. 

The <span style="color:green"><strong>connected</strong></span> text in the top right corner of the visualizer indicates you are actively communicating with your local session. If for some reason this <span style="color:red"><strong>disconnects</strong></span>, the visualizer will need to be reset as it will lose track of any changes.

Congrats! Pylabrobot is working as expected!

---

**TO-DO:** Take a screenshot of your deck setup and save it to submit with the Workshop 0 assignment on Canvas.

---

#### Creating a GIF of a lab protocol

We kinda added everything at once there, but ideally, we would like to create a visualization of any lab protocol we use, **step-by-step**. The classic way to do this with PLR is to click **Start Recording** in the visualizer before running your code, click **Stop Recording** afterwards, then type a filename and click **Download GIF**. That works, but it means tab-switching at exactly the right moments for every single deliverable -- and there are many deliverables in this course.

Instead, every workshop uses a small course helper that records **from your code**:

```python
rec = gif_recorder(lh.vis, name="my_protocol.gif")

await rec.start()
# ... deck changes and pipetting ...
await rec.stop()   # GIF renders and downloads automatically
```

Everything the deck does between `rec.start()` and `rec.stop()` is captured frame-by-frame, and the finished GIF **downloads automatically** under the exact filename you gave. No buttons, no tab switching, no renaming. A few things worth knowing:

1. `rec.start()` waits for the visualizer page to connect before recording begins, so you don't need artificial delays to go click a button first.

2. You still control pacing yourself: pauses between operations (`await asyncio.sleep(N * SLEEP)`, or `await step(N)` from the same module) are what make each state visible as its own frame.

3. If you forget `rec.stop()`, the recorder stops itself after **60 seconds** and downloads what it captured -- so a forgotten stop costs a short GIF, not a lost one.

4. The **Start/Stop Recording** toolbar buttons still work exactly as before if you ever want a manual, ad-hoc recording -- the two approaches don't interfere.

5. Generally, it is good practice to work first without the visualizer as much as possible. The reason is because if an error occurs while you are using the visualizer, you may have to **reset** both the **deck** and **visualizer**. Every time the visualizer is reset, it will open a **new tab**, which can become annoying after several iterations.

    - It is also good to **functionalize your code** to have one function which **sets up your deck** and one which **runs the protocol**. Typically you will find most of your errors will occur in coding the protocol, so having a function to easily set up your deck again will **speed up your dvelopment**

    - We will cover this more in the deck setup tutorial.


In [ ]:
import asyncio

# Course helpers. visualize_deck() builds the LiquidHandler + visualizer pair
# and attaches the visualizer as `lh.vis`; step() pauses using the SLEEP
# constant at the top of this notebook.
from bme590.visualizer_ext import gif_recorder, set_step_delay, step, visualize_deck

set_step_delay(SLEEP)

deck = STARLetDeck()
lh = await visualize_deck(deck, LiquidHandlerChatterboxBackend())

# IMPORTANT - the visualizer is set up before adding anything to the deck, so
# you can watch the deck grow piece by piece. Everything between rec.start()
# and rec.stop() below is recorded to lab_0_deck_setup.gif, which downloads
# automatically when you stop.
rec = gif_recorder(lh.vis, name="lab_0_deck_setup.gif")
await rec.start()

# create carriers
tip_carrier = TIP_CAR_480_A00(name="awesome tip carrier 96x5")
plate_carrier = PLT_CAR_L5AC_A00(name = "awesome plate carrier")

# assign carriers
deck.assign_child_resource(plate_carrier, rails = 5)
await step(0.5)
deck.assign_child_resource(tip_carrier, rails = 11)
await step(0.5)

# define 3 tip racks with unique names and assign to the first three slots of the tip carrier
for i in range(3):
    tip_carrier[i] = hamilton_96_tiprack_1000uL_filter(name=f"tip_rack_{i}")
    await step(0.5)

# define 4 plates with unique names and assign to the first four slots of the plate carrier
for i in range(4):
    plate_carrier[i] = cor_96_wellplate_360uL_Fb(name=f"plate_{i}")
    await step(0.5)

await rec.stop()
print("Done! lab_0_deck_setup.gif should appear in your Downloads folder.")


If done correctly, the deck was built piece by piece while being recorded, and `lab_0_deck_setup.gif` downloaded automatically to your browser's **Downloads folder** when `rec.stop()` ran. Open it and verify it shows the carriers, racks, and plates appearing step by step.

---

**TO-DO:** Submit `lab_0_deck_setup.gif` with the Workshop 0 assignment on Canvas. If the download did not start, make sure the visualizer tab was open and showing **connected** in the top right corner, then run the cell again.

---


#### Conclusion

That's all for workshop 0! As long as you were able to successfully download both a screenshot of your final deck setup and GIF, you should be good to go. 

For the remainder of the workshops, we will cover more technical aspects of PLR, but in general, all of the assignments will require you to submit one or more of the following:

1. A `.gif` file of your protocol running.

2. A `.png` or `.jpeg` file of your initial deck setup or final deck setup.

3. A write-up describing your protocol, and any extensions thereof.

4. A `.ipynb`, `.py`, or `.txt` file with your final code to produce that protocol.

For this assignment, you just need to submit the `.png` file and the `.gif` file. If you are still feeling unsure on how to generate any of the following, please **reach out to the teaching team**, contact info for whom can be found in the .`README.md` file on the [class GitHub](https://github.com/chory-lab/bme590-fall-2026)

---